# Incremental PCA

## 1. Introduction
Incremental PCA computes principal components in batches, reducing peak memory compared with fitting conventional PCA to all rows. Batch size must be at least the component count; results can differ slightly from batch PCA.

**Why / when:** Select the method based on data geometry, scale, operational constraints, and the question being asked. This lesson uses deterministic data and runs offline.


## 2. Mathematical intuition
Incremental PCA computes principal components in batches, reducing peak memory compared with fitting conventional PCA to all rows. Batch size must be at least the component count; results can differ slightly from batch PCA.

Let `n` be observations and `p` input features. The algorithm learns a structure from `X` without a target label; each method optimizes a different objective, so scores from different methods are not interchangeable.


## 3. Working mechanism
1. Inspect observations, feature units, missingness, and geometry. 2. Fit transformations on training data only. 3. Fit the unsupervised estimator without reference labels. 4. Inspect method-appropriate diagnostics and review with domain experts. 5. Persist the full fitted preprocessing/model artifact and monitor it.


## 4. Algorithm from scratch (NumPy)
This short PCA reference centers a matrix, applies singular value decomposition, and projects onto the leading right-singular vectors. It illustrates the linear algebra behind PCA; manifold methods use different objectives.


In [ ]:
import numpy as np
scratch_matrix = np.array([[2., 0.], [0., 1.], [1., 2.], [3., 1.]])
scratch_centered = scratch_matrix - scratch_matrix.mean(axis=0)
u, singular_values, vt = np.linalg.svd(scratch_centered, full_matrices=False)
scratch_embedding = scratch_centered @ vt[:2].T
print("singular values:", np.round(singular_values, 3))
print("2D PCA/SVD projection:\n", np.round(scratch_embedding, 3))


### Scratch dry run
Centering removes each feature mean. SVD returns directions ordered by singular value; projecting onto the first two directions gives a lower-dimensional representation.


## 5. Dataset and business context
The example uses a deterministic sample of scikit-learn's built-in digits data. Labels are only for visualization and are not passed to the reducer. In a real project, document data provenance, units, privacy, and the business meaning of a useful structure.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
TASK = 'reduction'
from sklearn.datasets import load_digits
digits = load_digits()
rng = np.random.default_rng(42)
indices = rng.choice(len(digits.data), size=500, replace=False)
X_raw = digits.data[indices].astype(float)
y_reference = digits.target[indices]
feature_names = [f"pixel_{i}" for i in range(X_raw.shape[1])]
X_df = pd.DataFrame(X_raw, columns=feature_names)
X_train_raw, X_test_raw, y_train_reference, y_test_reference = train_test_split(X_df, y_reference, test_size=0.2, random_state=42)
feature_names = list(X_df.columns)
print("feature matrix:", X_df.shape)
print("reference labels are excluded from fitting")


## 6. Exploratory data analysis
Review dimensions, feature distributions, missingness, duplicates, and pairwise structure. A visualization can reveal scale issues or data errors, but it does not prove that clusters or latent factors are real.


In [ ]:
print("missing values:", int(X_df.isna().sum().sum()))
print("duplicate rows:", int(X_df.duplicated().sum()))
print(X_df.describe().T.head(10))
plt.scatter(X_train_raw.iloc[:, 0], X_train_raw.iloc[:, 1], s=12, alpha=.7)
plt.title("First two raw features (exploration only)")
plt.show()


## 7. Preprocessing without leakage
Split observations first. Fit the scaler using training rows and reuse it for test/new data. Scaling matters for distance and neighborhood methods. NMF requires non-negative input.


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)
print("scaled train/test shapes:", X_train_scaled.shape, X_test_scaled.shape)


## 8. Baseline model
Use a deliberately simple reference method as a sanity check. The baseline does not use reference labels. Compare diagnostics and business usefulness rather than choosing an algorithm based on one score.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import trustworthiness
baseline = PCA(n_components=2, random_state=42).fit(X_train_scaled)
baseline_embedding = baseline.transform(X_train_scaled)
print("PCA baseline trustworthiness:", round(float(trustworthiness(X_train_scaled, baseline_embedding, n_neighbors=10)), 3))


## 9. Detailed training
Fit the algorithm-specific estimator on transformed training features only. Reference labels are never passed to `fit` or `fit_predict`.


In [ ]:
from sklearn.decomposition import IncrementalPCA
model = IncrementalPCA(n_components=2, batch_size=100)
embedding = model.fit_transform(X_train_scaled)
print("embedding shape:", embedding.shape)


## 10. Evaluation
Trustworthiness checks local-neighborhood preservation; reconstruction error is shown only when supported. These diagnostics are not held-out predictive metrics.


In [ ]:
from sklearn.manifold import trustworthiness
trust = float(trustworthiness(X_train_scaled, embedding, n_neighbors=10))
print("training-sample trustworthiness:", round(trust, 3))
if hasattr(model, "explained_variance_ratio_"):
    print("explained variance ratio:", np.round(model.explained_variance_ratio_, 3))
if hasattr(model, "inverse_transform") and model.get_params(deep=False).get("fit_inverse_transform", True):
    reconstructed = model.inverse_transform(embedding)
    print("reconstruction MSE:", round(float(np.mean((X_train_scaled - reconstructed) ** 2)), 4))
plt.figure(figsize=(8, 6))
scatter = plt.scatter(embedding[:, 0], embedding[:, 1], c=y_train_reference, s=14, cmap="tab10")
plt.colorbar(scatter, label="Digit label (visualization only)")
plt.title("Training embedding (labels did not train the model)")
plt.show()


## 11. Stability and generalization
Unsupervised models have no universal train/test accuracy. Check whether structure is stable under random seeds, resampling, and reasonable perturbations. Use label-invariant scores for clustering, or neighborhood preservation for embeddings.


In [ ]:
from sklearn.base import clone
from sklearn.manifold import trustworthiness
repeat = clone(model)
if "random_state" in repeat.get_params(deep=False):
    repeat.set_params(random_state=7)
repeat_embedding = repeat.fit_transform(X_train_scaled)
print("first-fit trustworthiness:", round(float(trustworthiness(X_train_scaled, embedding, n_neighbors=10)), 3))
print("repeat-fit trustworthiness:", round(float(trustworthiness(X_train_scaled, repeat_embedding, n_neighbors=10)), 3))


## 12. Hyperparameter optimization
Tune a small, meaningful parameter range using training-only diagnostics and domain context. Avoid repeatedly inspecting the final test partition. Clustering label identifiers can change between fits, so compare partitions with permutation-invariant measures.


In [ ]:
from sklearn.base import clone
from sklearn.manifold import trustworthiness
for dimensions in (2, 3):
    candidate = clone(model).set_params(n_components=dimensions)
    candidate_embedding = candidate.fit_transform(X_train_scaled)
    score = trustworthiness(X_train_scaled, candidate_embedding, n_neighbors=10)
    print(f"n_components={dimensions}: trustworthiness={score:.3f}")


## 13. Explainability and interpretation
Unsupervised structures are descriptive, not causal. Summaries can help characterize groups or dimensions aligned with an embedding. Validate interpretations with domain experts; do not overclaim from a visualization.


In [ ]:
associations = np.corrcoef(X_train_scaled.T, embedding.T)[:X_train_scaled.shape[1], X_train_scaled.shape[1]:]
associations = np.nan_to_num(associations)
for component in range(associations.shape[1]):
    order = np.argsort(np.abs(associations[:, component]))[::-1][:5]
    print(f"component {component + 1} strongest associations:")
    print([(feature_names[i], round(float(associations[i, component]), 3)) for i in order])
if hasattr(model, "components_"):
    print("component matrix shape:", model.components_.shape)


## 14. Advanced techniques
Compare a 3D representation using neighborhood preservation; do not compare raw coordinates across fits because embeddings may rotate or change sign.


In [ ]:
# Compare a 3D embedding using a local-neighborhood diagnostic.
from sklearn.base import clone
from sklearn.manifold import trustworthiness
if "n_components" in model.get_params(deep=False):
    model_3d = clone(model).set_params(n_components=3)
    embedding_3d = model_3d.fit_transform(X_train_scaled)
    print("3D embedding shape:", embedding_3d.shape)
    print("3D trustworthiness:", round(float(trustworthiness(X_train_scaled, embedding_3d, n_neighbors=10)), 3))
else:
    print("This estimator does not expose n_components.")


## 15. Production implementation
Package the fitted scaler and estimator together with feature order and metadata so inference uses the same preprocessing. Pickle is executable serialization: load only trusted, version-controlled artifacts.


In [ ]:
import json
import pickle
from pathlib import Path
import sklearn
artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)
model_path = artifact_dir / "unsupervised_model.pkl"
bundle = {"scaler": scaler, "model": model, "feature_names": feature_names, "task": TASK}
with model_path.open("wb") as handle:
    pickle.dump(bundle, handle)
metadata = {"algorithm": model.__class__.__name__, "task": TASK, "sklearn_version": sklearn.__version__, "feature_count": len(feature_names), "artifact": str(model_path)}
(artifact_dir / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
with model_path.open("rb") as handle:
    reloaded = pickle.load(handle)
print("saved:", model_path, "| reloaded features:", len(reloaded["feature_names"]))
print("smoke-test transformed shape:", reloaded["scaler"].transform(X_test_raw.iloc[:2]).shape)


## 16. Deployment
The FastAPI template loads the trusted bundle once at startup and validates feature names. Reducers such as t-SNE do not support general out-of-sample transform. The API returns HTTP 501 for those estimators rather than presenting a misleading inference path.


In [ ]:
import pickle
from pathlib import Path
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
with Path("artifacts/unsupervised_model.pkl").open("rb") as handle:
    bundle = pickle.load(handle)
app = FastAPI(title="Unsupervised ML API")
class AnalyzeRequest(BaseModel):
    features: dict[str, float]
@app.get("/health")
def health():
    return {"status": "ok"}
@app.post("/analyze")
def analyze(request: AnalyzeRequest):
    expected = bundle["feature_names"]
    if set(request.features) != set(expected):
        raise HTTPException(status_code=422, detail={"expected_features": expected})
    row = pd.DataFrame([[request.features[name] for name in expected]], columns=expected)
    transformed = bundle["scaler"].transform(row)
    if not hasattr(bundle["model"], "transform"):
        raise HTTPException(status_code=501, detail="This reducer has no out-of-sample transform; use a deployable reducer or validated batch-refit workflow.")
    transformed_rows = bundle["model"].transform(transformed)
    return {"embedding": transformed_rows.tolist()}
    raise HTTPException(status_code=501, detail="No supported inference operation.")


## 17. MLOps
Record algorithm and parameter versions, training-data identity, preprocessing, diagnostics, artifact checksum, and runtime dependencies. Monitor feature drift, cluster-size/noise changes or embedding transform failures, latency, memory, and downstream usefulness. Retraining and rollout should be reviewed and reversible.


In [ ]:
import hashlib
import json
from datetime import datetime, timezone
training_fingerprint = hashlib.sha256(np.ascontiguousarray(X_train_scaled).tobytes()).hexdigest()
run_record = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "algorithm": model.__class__.__name__,
    "parameters": {key: str(value) for key, value in model.get_params(deep=False).items()},
    "training_rows": int(len(X_train_scaled)),
    "feature_count": int(X_train_scaled.shape[1]),
    "training_data_fingerprint": training_fingerprint,
    "artifact": str(model_path),
    "trustworthiness": float(trust), "embedding_dimensions": int(embedding.shape[1]),

}
(artifact_dir / "run_metadata.json").write_text(json.dumps(run_record, indent=2), encoding="utf-8")
print(run_record)


## 18. Performance optimization
Measure on representative data and hardware. Report fit time separately from online inference latency. Python memory tracing does not capture every native-library allocation. Reduce dimensions or sample data only when that preserves the intended analysis.


In [ ]:
import time
import tracemalloc
from sklearn.base import clone
sample_size = min(200, len(X_train_scaled))
benchmark = clone(model)
tracemalloc.start()
start = time.perf_counter()
benchmark.fit_transform(X_train_scaled[:sample_size])
fit_seconds = time.perf_counter() - start
_, peak_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"fit time for {sample_size} rows: {fit_seconds:.4f} s")
print(f"fit throughput: {sample_size / fit_seconds:.1f} rows/s")
print(f"Python-traced peak: {peak_bytes / 1024:.1f} KB (native allocations may not be included)")


## 19. Production case study
**Workflow:** define the question and observation unit; verify point-in-time-correct features; establish preprocessing and validation; fit the method; review stability and diagnostics with domain experts; version model and data; deploy only a supported inference operation; monitor drift, output distribution, runtime, and downstream outcomes. Define rollback to a known-good artifact.


## 20. Interview preparation
1. Is this supervised? **No; target labels are not used in fitting.** 2. Why scale? **Distance and geometry can be scale-sensitive.** 3. What assumptions does it make? **See section 1.** 4. How do you select parameters? **Training-only diagnostics, stability, and domain context.** 5. How evaluate without labels? **Internal metrics and expert review; synthetic labels here are optional diagnostics.** 6. Can it process unseen rows? **Only with a supported predict/transform operation.** 7. How prevent leakage? **Fit preprocessing on training rows.** 8. What monitor? **Drift, output distribution, runtime, and usefulness.** 9. Can a 2D plot prove clusters? **No.** 10. How deploy safely? **Version trusted artifacts, validate schemas, and preserve rollback.**


## 21. Exercises
**Beginner:** change the seed and inspect results. **Intermediate:** compare two meaningful parameter settings and explain diagnostic changes. **Advanced:** test stability under resampling or compare another representation. **Production:** package preprocessing with the estimator and document whether unseen-row inference is supported. Never use reference labels to fit the unsupervised model.
